In [1]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from transformers_sae import _autoreload

import os
from transformers_sae.ops import MemoryTrackingMode, load_saes, method_to_saes
from transformers_sae.replacement_model import GemmaReplacement, make_replacement_model

# Could change this to replicate on a different suite of SAEs
STANDARD_SAES_FOR_ANALYSIS = "gemma_2_2b/gemma_scope_100_l0_tuned_encoder_0"
RA_SAES_FOR_ANALYSIS = "gemma_2_2b/next_layer_in_place_finetuned_lista_unit_scale"
GEMMA_SCOPE_SAES = "gemma_scope_canonical_l0"

LAYER_FOR_ANALYSIS = 15

# Tweak TRAINING_BATCH_SIZE for your hardware if necessary
if torch.cuda.is_available():
    TRAINING_DEVICE = "cuda:0"
    TRAINING_BATCH_SIZE = 2
elif torch.mps.is_available():
    TRAINING_DEVICE = "mps:0"
    TRAINING_BATCH_SIZE = 2
else:
    TRAINING_DEVICE = "cpu"
    TRAINING_BATCH_SIZE = 2

model_id = "google/gemma-2-2b"
tokenizer = AutoTokenizer.from_pretrained(model_id)
training_dataset = load_dataset(
    "monology/pile-uncopyrighted-parquet",
    split="train",
    streaming=True,
    columns=["text"],
)
validation_dataset = load_dataset(
    "monology/pile-test-val",
    split="validation",
    revision="refs/convert/parquet",
    streaming=True,
    columns=["text", "meta"],
)

with MemoryTrackingMode() as mtm:
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map=TRAINING_DEVICE,
        dtype=torch.bfloat16,
        use_safetensors=True,
    )
    model = make_replacement_model(
        model,
        {},
        num_layers=model.config.num_hidden_layers,
        context_length=1024,  # model.config.max_position_embeddings,
        d_model=model.config.hidden_size,
        layer_path="model.layers",
        replacement_class=GemmaReplacement,
    )
    model.eval()
    model.requires_grad_(False)

ra_saes = load_saes(
    f"{os.getenv('HF_BUCKET_LOCAL')}/{RA_SAES_FOR_ANALYSIS}", range(model.num_layers)
)
ra_replacement_model = make_replacement_model(model, ra_saes)
for sae in ra_saes.values():
    sae.eval()
    sae.onload()

standard_saes = load_saes(
    f"{os.getenv('HF_BUCKET_LOCAL')}/{STANDARD_SAES_FOR_ANALYSIS}",
    range(model.num_layers),
)
standard_replacement_model = make_replacement_model(model, standard_saes)
for sae in standard_saes.values():
    sae.eval()
    sae.onload()

gemma_scope_saes = method_to_saes(
    "", GEMMA_SCOPE_SAES, range(model.num_layers), TRAINING_DEVICE
)
for sae in gemma_scope_saes.values():
    sae.eval()
    sae.onload()

gemma_scope_replacement_model = make_replacement_model(model, gemma_scope_saes)

/cloud-dev/.venv/lib/python3.12/site-packages/codefind/registry.py:46: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, types.FunctionType):


Resolving data files:   0%|          | 0/1987 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1987 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Loading thresholds from /workspace/sae_checkpoints/gemma_2_2b/next_layer_in_place_finetuned_lista_unit_scale/train_thresholds_0
Loaded checkpoint for layer 10
Updated thresholds for layer 10
Loaded checkpoint for layer 23
Updated thresholds for layer 23
Loaded checkpoint for layer 22
Updated thresholds for layer 22
Loaded checkpoint for layer 14
Updated thresholds for layer 14
Loaded checkpoint for layer 16
Updated thresholds for layer 16
Loaded checkpoint for layer 21
Updated thresholds for layer 21
Loaded checkpoint for layer 11
Updated thresholds for layer 11
Loaded checkpoint for layer 15
Updated thresholds for layer 15
Loaded checkpoint for layer 17
Updated thresholds for layer 17
Loaded checkpoint for layer 24
Updated thresholds for layer 24
Loaded checkpoint for layer 25
Updated thresholds for layer 25
Loaded checkpoint for layer 7
Updated thresholds for layer 7
Loaded checkpoint for layer 13
Updated thresholds for layer 13
Loaded checkpoint for layer 12
Updated thresholds for l

In [7]:
from transformers_sae.metrics import kl_eval
from transformers_sae.ops import generate

# All but the final sentence of the first PubMed abstract
pile_example = (
    ".".join(
        next(
            validation_dataset.filter(
                lambda ex: ex["meta"]["pile_set_name"] == "PubMed Abstracts"
            ).iter(1)
        )["text"][0].split(".")[:-2]
    )
    + "."
)
final_sentence = "Our study strongly suggests that sleep disturbances, mainly shorter total sleep time, poor sleep efficiency, and prolonged sleep latencies, are associated with impaired memory and executive function in patients with refractory focal epilepsy and to a lesser extent, among those with medically controlled epilepsy."
true_continuation = tokenizer.decode(
    tokenizer(final_sentence, return_tensors="pt")["input_ids"][0, :10]
)
print("True continuation: ", true_continuation, "\n\n")

prompts = [
    pile_example,
]
max_tokens = 10

for p in prompts:
    print(p)
    print("Replacement-aware replacement model: ")
    generate(
        p,
        ra_replacement_model,
        tokenizer,
        max_new_tokens=max_tokens,
        do_sample=False,
    )
    print("Standard replacement model: ")
    generate(
        p,
        standard_replacement_model,
        tokenizer,
        max_new_tokens=max_tokens,
        do_sample=False,
    )

    print("Gemma Scope replacement model: ")
    generate(
        p,
        gemma_scope_replacement_model,
        tokenizer,
        max_new_tokens=max_tokens,
        do_sample=False,
    )
    print("\n\nBase model:")
    generate(p, model, tokenizer, max_new_tokens=max_tokens, do_sample=False)
    print("-----\n\n")

True continuation:  <bos>Our study strongly suggests that sleep disturbances, mainly 


Effect of sleep quality on memory, executive function, and language performance in patients with refractory focal epilepsy and controlled epilepsy versus healthy controls - A prospective study.
We aimed to evaluate the effect of sleep quality on memory, executive function, and language performance in patients with refractory focal epilepsy and controlled epilepsy and compare these with healthy individuals. We prospectively enrolled 37 adolescent and adult patients with refractory focal epilepsy (Group 1) and controlled epilepsy (Group 2) in each group. History pertaining to epilepsy and sleep were recorded, and all patients underwent overnight polysomnography. Language, memory, and executive function assessments were done using Western Aphasia Battery, Post Graduate Institute (PGI) memory scale, and battery of four executive function tests (Trail Making Test A & B, Digit symbol test, Stroop Task, an

In [8]:
from transformers_sae.metrics import kl_loss
from transformers_sae.tokenization import make_dataloader
from transformers_sae.activation_data import make_activation_batch

example_prompt_dataset = validation_dataset.filter(
    lambda ex: ex["meta"]["pile_set_name"] == "PubMed Abstracts"
).take(1)


for name, model_to_run in {
    "base": model,
    "gemma_scope": gemma_scope_replacement_model,
    "gemma_scope_encoder_tuned": standard_replacement_model,
    "replacement_aware": ra_replacement_model,
}.items():
    for batch in make_dataloader(
        model_to_run,
        tokenizer,
        example_prompt_dataset,
        max_tokens=None,
        tokenizer_batch_size=1,
        inference_batch_size=1,
    ):
        batch.to(model_to_run.device)
        input_args, input_kwargs = model_to_run.get_base_model_args(batch, None, True)
        with torch.no_grad():
            replacement_log_probs = make_activation_batch(
                model_to_run, [(model.num_layers, "layer")], batch
            )[model.num_layers].log_probs
            true_log_probs = make_activation_batch(
                model, [(model.num_layers, "layer")], batch
            )[model.num_layers].log_probs

            print(
                f"{name} KL: ",
                kl_loss(
                    replacement_log_probs, true_log_probs, batch, return_type="float"
                ),
            )

base KL:  9.74978320300579e-10
gemma_scope KL:  11.0625
gemma_scope_encoder_tuned KL:  1.1953125
replacement_aware KL:  0.359375


In [9]:
from transformers_sae.ops import generate

prompts = [
    "The capital of France,",
    "The capital of the United Kingdom,",
    "The capital of Japan,",
    "San Francisco, home of the iconic",
    "New York City, home of the iconic",
    "Paris, home of the iconic",
]
max_tokens = 10

for p in prompts:
    print(p)
    print("Replacement-aware replacement model: ")
    generate(
        p,
        ra_replacement_model,
        tokenizer,
        max_new_tokens=max_tokens,
        do_sample=False,
    )
    print("Standard replacement model: ")
    generate(
        p,
        standard_replacement_model,
        tokenizer,
        max_new_tokens=max_tokens,
        do_sample=False,
    )

    print("Gemma Scope replacement model: ")
    generate(
        p,
        gemma_scope_replacement_model,
        tokenizer,
        max_new_tokens=max_tokens,
        do_sample=False,
    )
    print("\n\nBase model:")
    generate(p, model, tokenizer, max_new_tokens=max_tokens, do_sample=False)
    print("-----\n\n")

The capital of France,
Replacement-aware replacement model: 
 Paris, is a beautiful city with a lot of
Standard replacement model: 
 the a11111111
Gemma Scope replacement model: 
   // //    


Base model:
 Paris, is a city that is known for its
-----


The capital of the United Kingdom,
Replacement-aware replacement model: 
 London, is the capital of the United Kingdom.
Standard replacement model: 
 the the UK government of the UK, is a
Gemma Scope replacement model: 
 of   //     


Base model:
 London, is a city that is full of history
-----


The capital of Japan,
Replacement-aware replacement model: 
 Tokyo, is a city of 1.2
Standard replacement model: 
 a country that is known most likely the most important
Gemma Scope replacement model: 
	 <pad>.      


Base model:
 Tokyo, is a city that is always bustling with
-----


San Francisco, home of the iconic
Replacement-aware replacement model: 
 San Francisco Valley, is a must-have destination
Standard replacement model: 
 and well-